In [1]:
import pandas as pd

df = pd.read_excel(r"C:\AI Project\AI_Resume_Screening.csv.xlsx")
print(df.shape)
print(df.columns.tolist())
df.head()

(1000, 11)
['Resume_ID', 'Name', 'Skills', 'Experience (Years)', 'Education', 'Certifications', 'Job Role', 'Recruiter Decision', 'Salary Expectation ($)', 'Projects Count', 'AI Score (0-100)']


,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


In [2]:
# Step 1: Import all libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Step 2: Load dataset
df = pd.read_excel(r"C:\AI Project\AI_Resume_Screening.csv.xlsx")
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

Dataset loaded successfully!
Shape: (1000, 11)


,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


In [3]:
!pip install streamlit scikit-learn PyPDF2 groq plotly

In [4]:
# checking data
print(df.isnull().sum())
print("\n")
print(df['Recruiter Decision'].value_counts())

Resume_ID                   0
Name                        0
Skills                      0
Experience (Years)          0
Education                   0
Certifications            274
Job Role                    0
Recruiter Decision          0
Salary Expectation ($)      0
Projects Count              0
AI Score (0-100)            0
dtype: int64


Recruiter Decision
Hire      812
Reject    188
Name: count, dtype: int64


In [5]:
# preparing data for model
le = LabelEncoder()

# converting text columns to numbers
df['Education_enc'] = le.fit_transform(df['Education'])
df['Job_Role_enc'] = le.fit_transform(df['Job Role'])

# target variable
df['Target'] = le.fit_transform(df['Recruiter Decision'])  # Hire=0, Reject=1

# features jo model use karega
features = ['Experience (Years)', 'AI Score (0-100)', 
            'Projects Count', 'Salary Expectation ($)', 
            'Education_enc', 'Job_Role_enc']

X = df[features]
y = df['Target']

print("Features ready!")
print(X.shape)

Features ready!
(1000, 6)


In [6]:
# splitting data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# training the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# checking accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {acc * 100:.2f}%")

Model Accuracy: 100.00%


In [7]:
import pickle

# saving the model
with open(r"C:\AI Project\model.pkl", "wb") as f:
    pickle.dump(model, f)

# saving label encoder bhi
with open(r"C:\AI Project\encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model saved successfully!")

Model saved successfully!


In [8]:
# ats score calculate karna based on resume data
def calculate_ats_score(experience, ai_score, projects, certifications):
    
    score = 0
    
    # experience ke basis pe
    if experience >= 5:
        score += 30
    elif experience >= 2:
        score += 20
    else:
        score += 10
    
    # ai score ke basis pe
    score += (ai_score * 0.4)
    
    # projects ke basis pe
    if projects >= 5:
        score += 20
    elif projects >= 2:
        score += 10
    else:
        score += 5
    
    # certifications ke basis pe
    if certifications != "None" and certifications != "":
        score += 10
    
    # 100 se zyada nahi hoga
    score = min(score, 100)
    
    return round(score, 2)

# testing
test_score = calculate_ats_score(3, 75, 4, "AWS")
print(f"ATS Score: {test_score}")


ATS Score: 70.0


In [9]:
# job recommend karna skills ke basis pe
def recommend_jobs(skills, experience):
    
    # job roles aur unki requirements
    job_data = {
        "Data Scientist": ["python", "machine learning", "deep learning", "tensorflow"],
        "Web Developer": ["html", "css", "javascript", "react"],
        "Android Developer": ["java", "kotlin", "android"],
        "ML Engineer": ["python", "machine learning", "pytorch", "nlp"],
        "Data Analyst": ["python", "sql", "excel", "tableau", "power bi"],
        "DevOps Engineer": ["docker", "kubernetes", "aws", "linux"],
        "Backend Developer": ["python", "java", "nodejs", "sql"],
        "AI Engineer": ["python", "deep learning", "nlp", "computer vision"]
    }
    
    skills_lower = skills.lower()
    recommendations = []
    
    for job, required_skills in job_data.items():
        match = 0
        for skill in required_skills:
            if skill in skills_lower:
                match += 1
        
        if match > 0:
            match_percent = round((match / len(required_skills)) * 100)
            recommendations.append((job, match_percent))
    
    # sort by match percentage
    recommendations.sort(key=lambda x: x[1], reverse=True)
    
    return recommendations[:3]  # top 3 jobs

# testing
jobs = recommend_jobs("Python, Machine Learning, TensorFlow", 3)
for job, match in jobs:
    print(f"{job} - {match}% match")

Data Scientist - 75% match
ML Engineer - 50% match
Backend Developer - 25% match


In [10]:
import PyPDF2
import re

# pdf se text nikalna
def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

# resume se skills dhundna
def extract_skills(text):
    common_skills = [
        "python", "java", "javascript", "react", "nodejs",
        "machine learning", "deep learning", "sql", "html", "css",
        "tensorflow", "pytorch", "docker", "kubernetes", "aws",
        "nlp", "computer vision", "tableau", "power bi", "excel",
        "kotlin", "android", "linux", "git", "mongodb"
    ]
    
    text_lower = text.lower()
    found_skills = []
    
    for skill in common_skills:
        if skill in text_lower:
            found_skills.append(skill)
    
    return found_skills

# resume se experience dhundna
def extract_experience(text):
    # years of experience pattern dhundna
    pattern = r'(\d+)\s*(?:years?|yrs?)'
    matches = re.findall(pattern, text.lower())
    if matches:
        return max([int(x) for x in matches])
    return 0

print("Resume parser ready!")

Resume parser ready!


In [11]:
from groq import Groq

client = Groq(api_key="YOUR_API_KEY_HERE")

def chat_with_ai(user_message, candidate_info=""):
    
    system_prompt = f"""
    You are an expert HR assistant and career coach.
    You help candidates with resume tips, interview preparation and career guidance.
    {f"Candidate Info: {candidate_info}" if candidate_info else ""}
    Keep answers short and helpful.
    """
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    
    return response.choices[0].message.content

reply = chat_with_ai("How can I improve my resume?")
print(reply)

To improve your resume, focus on:

1. Tailoring it to the job description
2. Using keywords from the industry
3. Highlighting achievements, not just responsibilities
4. Keeping it concise and easy to read
5. Including relevant sections like a professional summary and skills

Consider getting it reviewed by a career expert or using online resume builders for guidance.


In [12]:
from groq import Groq

client = Groq(api_key="YOUR_API_KEY_HERE")

models = client.models.list()
for model in models.data:
    print(model.id)

meta-llama/llama-guard-4-12b
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
moonshotai/kimi-k2-instruct-0905
openai/gpt-oss-safeguard-20b
meta-llama/llama-4-scout-17b-16e-instruct
meta-llama/llama-prompt-guard-2-86m
moonshotai/kimi-k2-instruct
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3-32b
groq/compound
groq/compound-mini
openai/gpt-oss-20b
llama-3.3-70b-versatile
allam-2-7b
whisper-large-v3
canopylabs/orpheus-arabic-saudi
meta-llama/llama-4-maverick-17b-128e-instruct
llama-3.1-8b-instant
whisper-large-v3-turbo


In [13]:
import plotly.express as px


skills_list = []
for skills in df['Skills']:
    for skill in str(skills).split(','):
        skills_list.append(skill.strip())

skills_df = pd.DataFrame(skills_list, columns=['Skill'])
top_skills = skills_df['Skill'].value_counts().head(10).reset_index()
top_skills.columns = ['Skill', 'Count']

fig = px.bar(top_skills, 
             x='Skill', 
             y='Count', 
             title='Top 10 Skills in Dataset',
             color='Skill')
fig.show()

In [14]:
# dataset skills
print(df['Skills'].head(20))

0                              TensorFlow, NLP, Pytorch
1          Deep Learning, Machine Learning, Python, SQL
2                 Ethical Hacking, Cybersecurity, Linux
3                           Python, Pytorch, TensorFlow
4                                      SQL, React, Java
5     Cybersecurity, Networking, Linux, Ethical Hacking
6            Networking, Cybersecurity, Ethical Hacking
7                              TensorFlow, Pytorch, NLP
8                           Networking, Ethical Hacking
9                      Python, TensorFlow, Pytorch, NLP
10                                       SQL, Java, C++
11           Cybersecurity, Ethical Hacking, Networking
12                            Cybersecurity, Networking
13                                       SQL, C++, Java
14                                         Pytorch, NLP
15                            Cybersecurity, Networking
16                                          SQL, Python
17    Linux, Networking, Cybersecurity, Ethical 

In [15]:
# graph 2 - hire vs reject
fig2 = px.pie(df, 
              names='Recruiter Decision',
              title='Hire vs Reject Ratio',
              color_discrete_sequence=['#00CC96', '#EF553B'])
fig2.show()

# graph 3 - experience distribution
fig3 = px.histogram(df, 
                    x='Experience (Years)',
                    title='Experience Distribution',
                    color='Recruiter Decision',
                    barmode='group')
fig3.show()

In [29]:
code = '''
import streamlit as st
import pandas as pd
import pickle
import plotly.express as px
from groq import Groq
import PyPDF2
import re

st.set_page_config(
    page_title="AI Resume Screening & Job Recommendation",
    page_icon="🤖",
    layout="wide"
)

st.markdown("""
<style>
    .main {background-color: #0e1117;}
    .stApp {background: linear-gradient(135deg, #0e1117 0%, #1a1f2e 100%);}
    
    .title-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 30px;
        border-radius: 15px;
        text-align: center;
        margin-bottom: 30px;
        box-shadow: 0 8px 32px rgba(102, 126, 234, 0.3);
    }
    
    div[data-testid="stMetric"] {
        background: linear-gradient(135deg, #1a1f2e, #2d3561);
        border: 1px solid #667eea;
        border-radius: 12px;
        padding: 15px;
        box-shadow: 0 4px 15px rgba(102, 126, 234, 0.2);
    }
    
    .stButton>button {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        border: none;
        border-radius: 8px;
        padding: 10px 25px;
        font-weight: bold;
    }
    
    .stButton>button:hover {
        transform: translateY(-2px);
        box-shadow: 0 5px 20px rgba(102, 126, 234, 0.4);
    }
    
    h1, h2, h3 {color: #ffffff;}
    p {color: #a0aec0;}
</style>
""", unsafe_allow_html=True)

client = Groq(api_key="your_api_key_here")

with open(r"C:\\\\AI Project\\\\model.pkl", "rb") as f:
    model = pickle.load(f)

with open(r"C:\\\\AI Project\\\\encoder.pkl", "rb") as f:
    le = pickle.load(f)

df = pd.read_excel(r"C:\\\\AI Project\\\\AI_Resume_Screening.csv.xlsx")

st.sidebar.markdown("""
<div style="text-align:center; padding: 20px 0;">
    <h2 style="color: #667eea;">🤖 AI Resume</h2>
    <p style="color: #a0aec0; font-size: 12px;">Screening & Job Recommendation</p>
</div>
""", unsafe_allow_html=True)

page = st.sidebar.selectbox("Navigation", 
    ["🏠 Home", "📄 Resume Screening", "💼 Job Recommendation", "📊 Dashboard", "💬 AI Chatbot"])

if page == "🏠 Home":
    st.markdown("""
    <div class="title-card">
        <h1 style="color: white; font-size: 2.5em;">🤖 AI Resume Screening</h1>
        <h3 style="color: #e0e0e0;">& Job Recommendation System</h3>
        <p style="color: #d0d0d0;">A smart AI-powered platform to screen resumes and find the perfect job match</p>
    </div>
    """, unsafe_allow_html=True)
    
    st.markdown("###  System Overview")
    col1, col2, col3, col4 = st.columns(4)
    col1.metric(" Total Resumes", len(df))
    col2.metric("✅ Hired", len(df[df["Recruiter Decision"] == "Hire"]))
    col3.metric("❌ Rejected", len(df[df["Recruiter Decision"] == "Reject"]))
    col4.metric("🎯 Model Accuracy", "100%")
    
    st.markdown("---")
    st.markdown("###  Key Features")
    c1, c2, c3 = st.columns(3)
    with c1:
        st.info(" **Resume Screening**\\n\\nUpload your PDF resume and get instant AI-powered screening with ATS score.")
    with c2:
        st.success("💼 **Job Recommendation**\\n\\nGet top job matches based on your skills and experience level.")
    with c3:
        st.warning(" **AI Career Coach**\\n\\nChat with our AI assistant for personalized career guidance and interview tips.")
'''
with open(r"C:\AI Project\app.py", "w", encoding="utf-8") as f:
    f.write(code)

print("app.py created successfully!")

app.py created successfully!


In [17]:
import subprocess
import sys

with open(r"C:\AI Project\app.py", "r", encoding="utf-8") as f:
    content = f.read()

if "login_user" in content:
    print("New code is saved correctly!")
else:
    print("Old code is running - need to save again!")

Old code is running - need to save again!


In [18]:
code = open(r"C:\AI Project\app.py", "r", encoding="utf-8").read()
print(code[:500])


import streamlit as st
import pandas as pd
import pickle
import plotly.express as px
from groq import Groq
import PyPDF2
import re

st.set_page_config(
    page_title="AI Resume Screening & Job Recommendation",
    page_icon="🤖",
    layout="wide"
)

st.markdown("""
<style>
    .main {background-color: #0e1117;}
    .stApp {background: linear-gradient(135deg, #0e1117 0%, #1a1f2e 100%);}

    .title-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 30


In [19]:
new_code = """
import streamlit as st
import pandas as pd
import pickle
import plotly.express as px
from groq import Groq
import PyPDF2
import re
import sqlite3
import hashlib

st.set_page_config(page_title="AI Resume Screening & Job Recommendation", page_icon="🤖", layout="wide")

st.markdown(\"\"\"
<style>
    .stApp {background: linear-gradient(135deg, #0e1117 0%, #1a1f2e 100%);}
    div[data-testid="stMetric"] {background: linear-gradient(135deg, #1a1f2e, #2d3561); border: 1px solid #667eea; border-radius: 12px; padding: 15px;}
    .stButton>button {background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border: none; border-radius: 8px; padding: 10px 25px; font-weight: bold; width: 100%;}
    .title-card {background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 15px; text-align: center; margin-bottom: 30px;}
    h1, h2, h3 {color: #ffffff;}
    p {color: #a0aec0;}
</style>
\"\"\", unsafe_allow_html=True)

def init_db():
    conn = sqlite3.connect(r"C:/AI Project/users.db")
    c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS users (username TEXT PRIMARY KEY, password TEXT, email TEXT)")
    conn.commit()
    conn.close()

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def signup_user(username, password, email):
    conn = sqlite3.connect(r"C:/AI Project/users.db")
    c = conn.cursor()
    try:
        c.execute("INSERT INTO users VALUES (?, ?, ?)", (username, hash_password(password), email))
        conn.commit()
        return True
    except:
        return False
    finally:
        conn.close()

def login_user(username, password):
    conn = sqlite3.connect(r"C:/AI Project/users.db")
    c = conn.cursor()
    c.execute("SELECT * FROM users WHERE username=? AND password=?", (username, hash_password(password)))
    result = c.fetchone()
    conn.close()
    return result is not None

init_db()

if "logged_in" not in st.session_state:
    st.session_state.logged_in = False
if "username" not in st.session_state:
    st.session_state.username = ""

with open(r"C:/AI Project/model.pkl", "rb") as f:
    model = pickle.load(f)
with open(r"C:/AI Project/encoder.pkl", "rb") as f:
    le = pickle.load(f)
df = pd.read_excel(r"C:/AI Project/AI_Resume_Screening.csv.xlsx")
client = Groq(api_key="YOUR_API_KEY_HERE")

def extract_text_from_pdf(uploaded_file):
    reader = PyPDF2.PdfReader(uploaded_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

def extract_skills(text):
    common_skills = ["python", "java", "javascript", "react", "nodejs", "machine learning",
        "deep learning", "sql", "html", "css", "tensorflow", "pytorch", "docker",
        "kubernetes", "aws", "nlp", "computer vision", "tableau", "power bi", "excel",
        "kotlin", "android", "linux", "git", "mongodb", "ethical hacking", "cybersecurity", "networking", "c++"]
    text_lower = text.lower()
    return [skill for skill in common_skills if skill in text_lower]

def calculate_ats_score(experience, ai_score, projects, certifications):
    score = 0
    if experience >= 5:
        score += 30
    elif experience >= 2:
        score += 20
    else:
        score += 10
    score += (ai_score * 0.4)
    if projects >= 5:
        score += 20
    elif projects >= 2:
        score += 10
    else:
        score += 5
    if certifications and certifications.lower() not in ["none", ""]:
        score += 10
    return round(min(score, 100), 2)

def recommend_jobs(skills):
    job_data = {
        "Data Scientist": ["python", "machine learning", "deep learning", "tensorflow"],
        "Web Developer": ["html", "css", "javascript", "react"],
        "Android Developer": ["java", "kotlin", "android"],
        "ML Engineer": ["python", "machine learning", "pytorch", "nlp"],
        "Data Analyst": ["python", "sql", "excel", "tableau", "power bi"],
        "DevOps Engineer": ["docker", "kubernetes", "aws", "linux"],
        "Backend Developer": ["python", "java", "nodejs", "sql"],
        "AI Engineer": ["python", "deep learning", "nlp", "computer vision"],
        "Cybersecurity Analyst": ["ethical hacking", "cybersecurity", "networking", "linux"],
        "Database Administrator": ["sql", "mongodb", "python"]
    }
    skills_lower = skills.lower()
    recommendations = []
    for job, required_skills in job_data.items():
        match = sum(1 for skill in required_skills if skill in skills_lower)
        if match > 0:
            match_percent = round((match / len(required_skills)) * 100)
            recommendations.append((job, match_percent))
    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:3]

def chat_with_ai(user_message):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an expert HR assistant and career coach. Help candidates with resume tips, interview preparation and career guidance. Keep answers short and helpful."},
            {"role": "user", "content": user_message}
        ]
    )
    return response.choices[0].message.content

def show_login():
    st.markdown(\"\"\"
    <div class="title-card">
        <h1 style="color:white;">🤖 AI Resume Screening</h1>
        <p style="color:#e0e0e0;">& Job Recommendation System</p>
    </div>
    \"\"\", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        tab1, tab2 = st.tabs(["Login", "Sign Up"])
        with tab1:
            st.markdown("### Welcome Back!")
            username = st.text_input("Username", key="login_user")
            password = st.text_input("Password", type="password", key="login_pass")
            if st.button("Login"):
                if login_user(username, password):
                    st.session_state.logged_in = True
                    st.session_state.username = username
                    st.rerun()
                else:
                    st.error("Invalid username or password!")
        with tab2:
            st.markdown("### Create Account")
            new_user = st.text_input("Username", key="signup_user")
            new_email = st.text_input("Email", key="signup_email")
            new_pass = st.text_input("Password", type="password", key="signup_pass")
            if st.button("Sign Up"):
                if signup_user(new_user, new_pass, new_email):
                    st.success("Account created! Please login.")
                else:
                    st.error("Username already exists!")

def show_main():
    st.sidebar.markdown(f\"\"\"
    <div style="text-align:center; padding:15px;">
        <h3 style="color:#667eea;">🤖 AI Resume</h3>
        <p style="color:#a0aec0;">Welcome, {st.session_state.username}!</p>
    </div>
    \"\"\", unsafe_allow_html=True)
    page = st.sidebar.selectbox("Navigation", ["Home", "Resume Screening", "Job Recommendation", "Dashboard", "AI Chatbot"])
    if st.sidebar.button("Logout"):
        st.session_state.logged_in = False
        st.rerun()

    if page == "Home":
        st.markdown(\"\"\"
        <div class="title-card">
            <h1 style="color:white;">🤖 AI Resume Screening & Job Recommendation</h1>
            <p style="color:#e0e0e0;">A smart AI-powered platform to screen resumes and find the perfect job match</p>
        </div>
        \"\"\", unsafe_allow_html=True)
        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Total Resumes", len(df))
        col2.metric("Hired", len(df[df["Recruiter Decision"] == "Hire"]))
        col3.metric("Rejected", len(df[df["Recruiter Decision"] == "Reject"]))
        col4.metric("Model Accuracy", "100%")
        st.markdown("---")
        c1, c2, c3 = st.columns(3)
        with c1:
            st.info("📄 Resume Screening - Upload your PDF resume and get instant AI screening with ATS score.")
        with c2:
            st.success("💼 Job Recommendation - Get top job matches based on your skills and experience.")
        with c3:
            st.warning("💬 AI Career Coach - Chat with AI for personalized career guidance.")

    elif page == "Resume Screening":
        st.markdown("## 📄 Resume Screening")
        col1, col2 = st.columns(2)
        with col1:
            uploaded_file = st.file_uploader("Upload Resume (PDF)", type=["pdf"])
            experience = st.slider("Years of Experience", 0, 20, 2)
            projects = st.number_input("Number of Projects", 0, 50, 3)
        with col2:
            education = st.selectbox("Education Level", ["Bachelor", "Master", "PhD", "Diploma"])
            certifications = st.text_input("Certifications (e.g. AWS, Google)")
            job_role = st.selectbox("Applying For", df["Job Role"].unique())
        skills_input = st.text_input("Your Skills (comma separated)", "Python, Machine Learning, SQL")
        if st.button("Screen My Resume"):
            if uploaded_file:
                pdf_text = extract_text_from_pdf(uploaded_file)
                extracted_skills = extract_skills(pdf_text)
                st.info(f"Skills extracted from PDF: {', '.join(extracted_skills) if extracted_skills else 'No common skills found'}")
            ai_score = 75
            ats_score = calculate_ats_score(experience, ai_score, projects, certifications)
            edu_enc = 0
            try:
                edu_enc = le.transform([education])[0]
            except:
                edu_enc = 0
            role_enc = 0
            try:
                role_enc = le.transform([job_role])[0]
            except:
                role_enc = 0
            features = [[experience, ai_score, projects, 50000, edu_enc, role_enc]]
            prediction = model.predict(features)[0]
            result = "Hire" if prediction == 0 else "Reject"
            st.markdown("---")
            st.markdown("### Screening Results")
            r1, r2, r3 = st.columns(3)
            r1.metric("ATS Score", f"{ats_score}/100")
            r2.metric("Screening Decision", result)
            r3.metric("Skills Found", len(skills_input.split(",")))
            if result == "Hire":
                st.success("Congratulations! Your profile looks strong. You are likely to be hired!")
            else:
                st.error("Your profile needs improvement. Consider adding more skills and projects.")
            fig = px.bar(x=["Experience", "AI Score", "Projects", "Certifications"],
                        y=[min(experience*6, 30), ai_score*0.4, min(projects*4, 20), 10 if certifications else 0],
                        title="ATS Score Breakdown", color_discrete_sequence=["#667eea"])
            st.plotly_chart(fig, use_container_width=True)

    elif page == "Job Recommendation":
        st.markdown("## 💼 Job Recommendation")
        skills = st.text_input("Enter Your Skills", "Python, Machine Learning, SQL")
        experience = st.slider("Years of Experience", 0, 20, 2)
        if st.button("Get Job Recommendations"):
            recommendations = recommend_jobs(skills)
            if recommendations:
                st.markdown("### Top Job Matches for You")
                for job, match in recommendations:
                    st.progress(match/100)
                    st.markdown(f"**{job}** — {match}% Match")
                    st.markdown("---")
                fig = px.bar(x=[r[0] for r in recommendations], y=[r[1] for r in recommendations],
                            title="Job Match Percentage", color=[r[1] for r in recommendations],
                            color_continuous_scale="purples")
                st.plotly_chart(fig, use_container_width=True)
            else:
                st.warning("No matching jobs found. Try adding more skills.")

    elif page == "Dashboard":
        st.markdown("## 📊 Dashboard")
        col1, col2 = st.columns(2)
        with col1:
            fig1 = px.pie(df, names="Recruiter Decision", title="Hire vs Reject Ratio",
                         color_discrete_sequence=["#667eea", "#764ba2"])
            st.plotly_chart(fig1, use_container_width=True)
        with col2:
            skills_list = []
            for s in df["Skills"]:
                for skill in str(s).split(","):
                    skills_list.append(skill.strip())
            skills_df = pd.DataFrame(skills_list, columns=["Skill"])
            top_skills = skills_df["Skill"].value_counts().head(10).reset_index()
            top_skills.columns = ["Skill", "Count"]
            fig2 = px.bar(top_skills, x="Skill", y="Count", title="Top 10 Skills", color="Skill")
            st.plotly_chart(fig2, use_container_width=True)
        fig3 = px.histogram(df, x="Experience (Years)", color="Recruiter Decision",
                           title="Experience Distribution", barmode="group",
                           color_discrete_sequence=["#667eea", "#764ba2"])
        st.plotly_chart(fig3, use_container_width=True)

    elif page == "AI Chatbot":
        st.markdown("## 💬 AI Career Coach")
        st.markdown("Ask anything about resumes, interviews, or career guidance.")
        if "messages" not in st.session_state:
            st.session_state.messages = []
        for msg in st.session_state.messages:
            with st.chat_message(msg["role"]):
                st.write(msg["content"])
        user_input = st.chat_input("Ask your career question...")
        if user_input:
            st.session_state.messages.append({"role": "user", "content": user_input})
            with st.chat_message("user"):
                st.write(user_input)
            reply = chat_with_ai(user_input)
            st.session_state.messages.append({"role": "assistant", "content": reply})
            with st.chat_message("assistant"):
                st.write(reply)

if st.session_state.logged_in:
    show_main()
else:
    show_login()
"""

with open(r"C:/AI Project/app.py", "w", encoding="utf-8") as f:
    f.write(new_code)

print("app.py saved successfully!")

app.py saved successfully!


In [20]:
import sqlite3
import hashlib

# check karo database mein kya stored hai
conn = sqlite3.connect(r"C:/AI Project/users.db")
c = conn.cursor()
c.execute("SELECT * FROM users")
rows = c.fetchall()
for row in rows:
    print(row)
conn.close()

('admin', '240be518fabd2724ddb6f04eeb1da5967448d7e831c08c8fa822809f74c720a9', 'admin@gmail.com')


In [21]:
import hashlib

password = "VAIBHAV123"
hashed = hashlib.sha256(password.encode()).hexdigest()
print("Generated hash:", hashed)
print("Stored hash:   ", "730680676f12736adfda3349ef61ec9e15cfbb3170da3d950ea56f13cd901542")
print("Match:", hashed == "730680676f12736adfda3349ef61ec9e15cfbb3170da3d950ea56f13cd901542")

Generated hash: 730680676f12736adfda3349ef61ec9e15cfbb3170da3d950ea56f13cd901542
Stored hash:    730680676f12736adfda3349ef61ec9e15cfbb3170da3d950ea56f13cd901542
Match: True


In [22]:
import sqlite3
import hashlib

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

conn = sqlite3.connect(r"C:/AI Project/users.db")
c = conn.cursor()
c.execute("SELECT * FROM users WHERE username=? AND password=?", 
          ("VAIBHAV_2602", hash_password("VAIBHAV123")))
result = c.fetchone()
conn.close()
print("Result:", result)

Result: None


In [23]:
import sqlite3
import hashlib

conn = sqlite3.connect(r"C:/AI Project/users.db")
c = conn.cursor()
c.execute("SELECT username FROM users")
rows = c.fetchall()
for row in rows:
    print(row)
conn.close()

('admin',)


In [24]:
import sqlite3

conn = sqlite3.connect(r"C:/AI Project/users.db")
c = conn.cursor()
c.execute("DELETE FROM users")
conn.commit()
conn.close()
print("Database cleared!")

Database cleared!


In [25]:
import sqlite3
import hashlib

conn = sqlite3.connect(r"C:/AI Project/users.db")
c = conn.cursor()

username = "admin"
password = "admin123"
email = "admin@gmail.com"
hashed = hashlib.sha256(password.encode()).hexdigest()

c.execute("DELETE FROM users")  # purane sab clear
c.execute("INSERT INTO users VALUES (?, ?, ?)", (username, hashed, email))
conn.commit()
conn.close()
print("User created!")
print(f"Username: {username}")
print(f"Password: {password}")

User created!
Username: admin
Password: admin123


In [ ]:
!streamlit run "C:/AI Project/app.py"

In [33]:
readme = """# AI Resume Screening & Job Recommendation System

## What is this project?

I built this project to solve a real problem that most companies face during hiring — manually going through hundreds of resumes takes a lot of time and is often inconsistent. This system automates that entire process using machine learning and AI.

You can upload your resume, get an ATS score, see which jobs match your profile, and even chat with an AI career coach — all in one place.

---

## Features

- Login and Signup system so each user has their own account
- Upload your PDF resume and get it screened automatically
- ATS score calculated based on your experience, projects, skills and certifications
- Job recommendations based on what skills you have
- Dashboard with charts showing hiring trends and skill distribution
- AI chatbot to help with career advice and interview tips

---

## Tech Stack

- **Python** — everything is built in Python
- **Streamlit** — for the web interface
- **Scikit-learn (Random Forest)** — the ML model that predicts hiring decisions
- **Groq API with LLaMA 3.3** — powers the AI chatbot
- **PyPDF2** — reads and extracts text from uploaded PDF resumes
- **Plotly** — for the interactive graphs and charts
- **SQLite** — stores user login data
- **Pandas** — for data handling and processing

---

## Project Structure
```
AI-Resume-Screening/
│
├── app.py                            # main application file
├── model.pkl                         # trained random forest model
├── encoder.pkl                       # label encoder for categories
├── AI_Resume_Screening.csv.xlsx      # dataset used for training
├── users.db                          # stores user accounts
└── README.md                         # this file
```

---

## How to Run

**Step 1 — Clone the repo**
```bash
git clone https://github.com/yourusername/ai-resume-screening.git
cd ai-resume-screening
```

**Step 2 — Install the required libraries**
```bash
pip install streamlit scikit-learn PyPDF2 groq plotly pandas openpyxl
```

**Step 3 — Add your Groq API key**

Go to [console.groq.com](https://console.groq.com), create a free account and generate an API key.
Open `app.py` and replace `YOUR_API_KEY_HERE` with your actual key.

**Step 4 — Run the app**
```bash
streamlit run app.py
```

---

## Dataset Details

The dataset has 1000 resume records with the following columns — Resume ID, Name, Skills, Experience in Years, Education, Certifications, Job Role, Recruiter Decision, Salary Expectation, Projects Count, and AI Score.

The Random Forest model trained on this dataset gives 100% accuracy on the test set.

---

## How It Works

**Resume Screening**
Upload your PDF resume or manually enter your details. The system extracts your skills, calculates an ATS score, and predicts whether your profile would likely get a hire or reject decision.

**Job Recommendation**
Enter your skills and the system matches them against common job roles. You get the top 3 job matches along with a match percentage for each.

**AI Chatbot**
Ask anything related to your career — resume tips, how to prepare for interviews, what skills to learn next. The chatbot is powered by LLaMA 3.3 through Groq and gives practical advice.

---

## What I Learned

This project taught me how to connect machine learning models with a proper frontend, how to handle PDF parsing, how to use LLM APIs in a real application, and how to manage user sessions and authentication without any heavy frameworks.

---

## Possible Improvements

- Better resume parsing using NLP instead of keyword matching
- Adding more job categories
- Letting recruiters log in separately and view candidate reports
- Sending email notifications after screening
- Deploying it online so anyone can use it

---

## Author

**Vaibhav Dubey**
BTech CS AI — Kanpur

GitHub: [Vaibhavd2602](https://github.com/Vaibhavd2602)
"""

with open(r"C:/AI Project/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md created!")

README.md created!


In [35]:
!git --version

git version 2.53.0.windows.1


In [36]:
import os
os.chdir(r"C:/AI Project")

!git init
!git add .
!git commit -m "AI Resume Screening and Job Recommendation System"
!git branch -M main
!git remote add origin https://github.com/Vaibhavd2602/ai-resume-screening.git
!git push -u origin main

Initialized empty Git repository in C:/AI Project/.git/


[master (root-commit) adab97a] AI Resume Screening and Job Recommendation System
 10 files changed, 11192 insertions(+)
 create mode 100644 .ipynb_checkpoints/Ai Project-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/TechStack-checkpoint.txt
 create mode 100644 AI_Resume_Screening.csv.xlsx
 create mode 100644 Ai Project.ipynb
 create mode 100644 README.md
 create mode 100644 TechStack.txt
 create mode 100644 app.py
 create mode 100644 encoder.pkl
 create mode 100644 model.pkl
 create mode 100644 users.db


fatal: User cancelled dialog.
bash: line 1: /dev/tty: No such device or address
error: failed to execute prompt script (exit code 1)
fatal: could not read Username for 'https://github.com': No such file or directory


In [42]:
!git push https://Vaibhavd2602@github.com/Vaibhavd2602/ai-resume-screening.git main

remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       â€”â€” Groq API Key â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”        
remote:        locations:        
remote:          - commit: adab97aa6a928d8ed4ff9ab64f64025f9bb51928

In [43]:
!git push -u origin main

remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       â€”â€” Groq API Key â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”        
remote:        locations:        
remote:          - commit: adab97aa6a928d8ed4ff9ab64f64025f9bb51928

In [46]:
import os
os.chdir(r"C:/AI Project")

!git add .
!git commit -m "removed api key from files"
!git push -u origin main

[main 129e012] removed api key from files
 1 file changed, 161 insertions(+), 5 deletions(-)


remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       â€”â€” Groq API Key â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”â€”        
remote:        locations:        
remote:          - commit: adab97aa6a928d8ed4ff9ab64f64025f9bb51928

In [47]:
!git push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/Vaibhavd2602/ai-resume-screening.git
 * [new branch]      main -> main


In [48]:
# .streamlit folder banana
import os
os.makedirs(r"C:/AI Project/.streamlit", exist_ok=True)

# secrets file banana
secrets = '[groq]\napi_key = "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"'

with open(r"C:/AI Project/.streamlit/secrets.toml", "w") as f:
    f.write(secrets)

print("Done!")

Done!


In [ ]:
with open(r"C:/AI Project/app.py", "r", encoding="utf-8") as f:
    content = f.read()

content = content.replace(
    'client = Groq(api_key="YOUR_API_KEY_HERE")',
    'client = Groq(api_key=st.secrets["groq"]["api_key"])'
)

with open(r"C:/AI Project/app.py", "w", encoding="utf-8") as f:
    f.write(content)

print("Done!")